In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from functools import reduce
from delta.tables import DeltaTable
from datetime import datetime, timezone

In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("batch_id", "2025-01-15")
dbutils.widgets.dropdown("run_mode", "initial", ["initial", "incremental"])
dbutils.widgets.text("job_run_id", "")

environment = dbutils.widgets.get("environment")
batch_id = dbutils.widgets.get("batch_id")
run_mode = dbutils.widgets.get("run_mode")
job_run_id = dbutils.widgets.get("job_run_id")

pipeline_name = "education_qa_pipeline"

if job_run_id:
    run_id = f"{environment}_{pipeline_name}_{batch_id}_{run_mode}_job_{job_run_id}"
else:
    run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"{environment}_{pipeline_name}_{batch_id}_{run_mode}_{run_timestamp}"

if run_mode == "initial":
    gold_write_mode = "overwrite"
elif run_mode == "incremental":
    gold_write_mode = "merge"
else:
    raise ValueError(f"Unsupported run_mode: {run_mode}")

catalog = "dbw_edu_qa_dev"
storage_account = "steduqadblakehouse"
container = "education-data-lake"

lake_root = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

print(f"environment: {environment}")
print(f"batch_id: {batch_id}")
print(f"run_mode: {run_mode}")
print(f"job_run_id: {job_run_id}")
print(f"run_id: {run_id}")
print(f"gold_write_mode: {gold_write_mode}")

environment: dev
batch_id: 2026-01-15
run_mode: incremental
job_run_id: 
run_id: dev_education_qa_pipeline_2026-01-15_incremental_20260608T054238Z
gold_write_mode: merge


In [0]:
source_tables = [
    "silver.schools",
    "silver.students",
    "silver.attendance",
    "silver.assessment_results",
    "silver.school_events",
    "qa.dq_validation_results",
    "qa.dq_failed_records",
    "qa.defect_log"
]

for table_name in source_tables:
    print(f"\n=== {catalog}.{table_name} ===")
    spark.table(f"{catalog}.{table_name}").printSchema()


=== dbw_edu_qa_dev.silver.schools ===
root
 |-- school_id: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- school_type: string (nullable = true)
 |-- open_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_record_id: string (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)


=== dbw_edu_qa_dev.silver.students ===
root
 |-- student_id: string (nullable = true)
 |-- school_id: string (nullable = true)
 |-- year_level: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- enrolment_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file

In [0]:
def write_delta_table(df, table_name, target_path, merge_condition=None):
    full_table_name = f"{catalog}.{table_name}"

    if gold_write_mode == "overwrite":
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("path", target_path)
            .saveAsTable(full_table_name)
        )

    elif gold_write_mode == "append":
        (
            df.write
            .format("delta")
            .mode("append")
            .option("path", target_path)
            .saveAsTable(full_table_name)
        )

    elif gold_write_mode == "merge":
        if not spark.catalog.tableExists(full_table_name):
            (
                df.write
                .format("delta")
                .mode("overwrite")
                .option("path", target_path)
                .saveAsTable(full_table_name)
            )
        else:
            # gets an existing Delta table from the catalog.
            delta_table = DeltaTable.forName(spark, full_table_name)

            (
                delta_table.alias("target")
                .merge(
                    df.alias("source"),
                    merge_condition
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

    else:
        raise ValueError(f"Unsupported gold_write_mode: {gold_write_mode}")

## Dimension tables
### dim_batch, dim_date, dim_school, dim_student, dim_year_level, dim_assessment_domain, dim_proficiency_band, dim_dq_rule

In [0]:
# gold.dim_batch
# Grain: one row per source batch.

batch_df = spark.createDataFrame(
    [
        ("2025-01-15", 1, "Batch 1 - 2024 data", 2024, "Initial load"),
        ("2026-01-15", 2, "Batch 2 - 2025 data", 2025, "Incremental load"),
    ],
    ["batch_id", "batch_sequence", "batch_label", "data_period_year", "batch_load_type"]
)

dim_batch_df = (
    batch_df
    .filter(F.col("batch_id") == batch_id)
    .withColumn(
        "batch_key",
        F.sha2(F.col("batch_id"), 256)
    )
    .withColumn("environment", F.lit(environment))
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "batch_key",
        "batch_id",
        "batch_sequence",
        "batch_label",
        "data_period_year",
        "batch_load_type",
        "environment",
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/dim_batch"

merge_condition = """
target.batch_id = source.batch_id
"""

write_delta_table(
    df=dim_batch_df,
    table_name="gold.dim_batch",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.dim_batch")
    .orderBy("batch_sequence")
)

batch_key,batch_id,batch_sequence,batch_label,data_period_year,batch_load_type,environment,gold_load_timestamp
399034112cd82562f0d651bda8a8b5ab8840703ee0b40cd136d85181164d2280,2025-01-15,1,Batch 1 - 2024 data,2024,Initial load,dev,2026-06-08T03:56:41.150Z
b94a975b4c4909c341e1e21579d655a38839844f91296c2e6baff25104d2c748,2026-01-15,2,Batch 2 - 2025 data,2025,Incremental load,dev,2026-06-08T04:03:13.875Z


In [0]:
# gold.dim_date
# Grain: one row per calendar date needed by attendance, assessment, school, student, and event data.

date_sources = []

date_sources.append(
    spark.table(f"{catalog}.silver.attendance")
    .filter(F.col("batch_id") == batch_id)
    .select(
        F.col("attendance_month").alias("date_value")
    )
)

date_sources.append(
    spark.table(f"{catalog}.silver.assessment_results")
    .filter(F.col("batch_id") == batch_id)
    .select(
        F.to_date(
            F.concat(F.col("assessment_year").cast("string"), F.lit("-01-01"))
        )
        .alias("date_value")
    )
)

# date_sources.append(
#     spark.table(f"{catalog}.silver.schools")
#     .filter(F.col("batch_id") == batch_id)
#     .select(
#         F.col("open_date").alias("date_value")
#     )
# )

# date_sources.append(
#     spark.table(f"{catalog}.silver.students")
#     .filter(F.col("batch_id") == batch_id)
#     .select(F.col("enrolment_date").alias("date_value"))
# )

date_sources.append(
    spark.table(f"{catalog}.silver.school_events")
    .filter(F.col("batch_id") == batch_id)
    .select(F.col("event_date").alias("date_value"))
)


date_bounds_df = (
    reduce(lambda df1, df2: df1.unionByName(df2), date_sources)
    .filter(F.col("date_value").isNotNull())
    .agg(
        F.min("date_value").alias("min_date"),
        F.max("date_value").alias("max_date")
    )
)

dim_date_df = (
    date_bounds_df
    .select(F.explode(F.sequence(F.col("min_date"), F.col("max_date"))).alias("full_date"))
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("calendar_year", F.year("full_date"))
    .withColumn("quarter_number", F.quarter("full_date"))
    .withColumn("month_number", F.month("full_date"))
    .withColumn("month_name", F.monthname("full_date"))
    .withColumn("month_start_date", F.trunc("full_date", "month"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_of_week_number", F.dayofweek("full_date"))
    .withColumn("day_of_week_name", F.date_format("full_date", "E"))
    .withColumn(
        "is_month_start",
        F.col("full_date") == F.col("month_start_date")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "date_key",
        "full_date",
        "calendar_year",
        "quarter_number",
        "month_number",
        "month_name",
        "month_start_date",
        "day_of_month",
        "day_of_week_number",
        "day_of_week_name",
        "is_month_start",
        "gold_load_timestamp"
    )    
)
   

target_path = f"{lake_root}/gold/dim_date"

merge_condition = """
target.date_key = source.date_key
"""

write_delta_table(
    df=dim_date_df,
    table_name="gold.dim_date",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.dim_date")
    .orderBy("full_date")
    .limit(50)
)

date_key,full_date,calendar_year,quarter_number,month_number,month_name,month_start_date,day_of_month,day_of_week_number,day_of_week_name,is_month_start,gold_load_timestamp
20240101,2024-01-01,2024,1,1,Jan,2024-01-01,1,2,Mon,true,2026-06-08T03:56:46.794Z
20240102,2024-01-02,2024,1,1,Jan,2024-01-01,2,3,Tue,false,2026-06-08T03:56:46.794Z
20240103,2024-01-03,2024,1,1,Jan,2024-01-01,3,4,Wed,false,2026-06-08T03:56:46.794Z
20240104,2024-01-04,2024,1,1,Jan,2024-01-01,4,5,Thu,false,2026-06-08T03:56:46.794Z
20240105,2024-01-05,2024,1,1,Jan,2024-01-01,5,6,Fri,false,2026-06-08T03:56:46.794Z
20240106,2024-01-06,2024,1,1,Jan,2024-01-01,6,7,Sat,false,2026-06-08T03:56:46.794Z
20240107,2024-01-07,2024,1,1,Jan,2024-01-01,7,1,Sun,false,2026-06-08T03:56:46.794Z
20240108,2024-01-08,2024,1,1,Jan,2024-01-01,8,2,Mon,false,2026-06-08T03:56:46.794Z
20240109,2024-01-09,2024,1,1,Jan,2024-01-01,9,3,Tue,false,2026-06-08T03:56:46.794Z
20240110,2024-01-10,2024,1,1,Jan,2024-01-01,10,4,Wed,false,2026-06-08T03:56:46.794Z


In [0]:
display(
    spark.table(f"{catalog}.gold.dim_date")
    .agg(
        F.count("*").alias("date_rows"),
        F.min("full_date").alias("min_date"),
        F.max("full_date").alias("max_date")
    )
)

date_rows,min_date,max_date
697,2024-01-01,2025-12-15


In [0]:
# gold.dim_school
# Grain: one row per school version.
# SCD Type 2 behaviour:
# - unchanged schools are not reinserted
# - changed schools expire the old current row and insert a new current row
# - new schools are inserted as current rows

from delta.tables import DeltaTable

source_school_df = (
    spark.table(f"{catalog}.silver.schools")
    .filter(F.col("batch_id") == batch_id)
)

duplicate_school_count = (
    source_school_df
    .groupBy("school_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_school_count > 0:
    raise ValueError("Duplicate school_id values found in the incoming school batch.")

incoming_school_df = (
    source_school_df
    .select(
        "school_id",
        "school_name",
        "region",
        "school_type",
        "status",
        "open_date",
        "bronze_record_id",
        "source_file_name",
        "run_id",
        "silver_load_timestamp"
    )
    .withColumn(
        "school_key",
        F.sha2(F.coalesce(F.col("school_id"), F.lit("")), 256)
    )
    .withColumn(
        "school_attribute_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("school_name"), F.lit("")),
                F.coalesce(F.col("region"), F.lit("")),
                F.coalesce(F.col("school_type"), F.lit("")),
                F.coalesce(F.col("status"), F.lit("")),
                F.coalesce(F.col("open_date").cast("string"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("effective_from_batch_id", F.lit(batch_id))
    .withColumn("effective_from_date", F.to_date(F.lit(batch_id)))
    .withColumn("effective_to_date", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
    .withColumn(
        "school_scd_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("school_id"), F.lit("")),
                F.coalesce(F.col("effective_from_batch_id"), F.lit(""))
            ),
            256
        )
    )
    .withColumn(
        "is_active_school",
        F.when(F.col("status") == "Active", F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "school_scd_key",
        "school_key",
        "school_id",
        "school_name",
        "region",
        "school_type",
        "status",
        "is_active_school",
        "open_date",
        "school_attribute_hash",
        "effective_from_batch_id",
        "effective_from_date",
        "effective_to_date",
        "is_current",
        "bronze_record_id",
        "source_file_name",
        "run_id",
        "silver_load_timestamp",
        "gold_load_timestamp"
    )
)

full_table_name = f"{catalog}.gold.dim_school"
target_path = f"{lake_root}/gold/dim_school"

if run_mode == "initial":
    (
        incoming_school_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("path", target_path)
        .saveAsTable(full_table_name)
    )

elif run_mode == "incremental":
    current_school_df = (
        spark.table(full_table_name)
        .filter(F.col("is_current") == True)
        .select(
            "school_key",
            "school_attribute_hash"
        )
    )

    changed_or_new_school_df = (
        incoming_school_df.alias("source")
        .join(
            current_school_df.alias("target"),
            on="school_key",
            how="left"
        )
        .filter(
            F.col("target.school_key").isNull()
            | (F.col("source.school_attribute_hash") != F.col("target.school_attribute_hash"))
        )
        .select("source.*")
    )

    changed_existing_school_keys_df = (
        changed_or_new_school_df.alias("source")
        .join(
            current_school_df.alias("target"),
            on="school_key",
            how="inner"
        )
        .select("school_key")
        .distinct()
    )

    changed_or_new_count = changed_or_new_school_df.count()

    if changed_or_new_count > 0:
        delta_table = DeltaTable.forName(spark, full_table_name)

        (
            delta_table.alias("target")
            .merge(
                changed_existing_school_keys_df.alias("source"),
                """
                target.school_key = source.school_key
                AND target.is_current = true
                """
            )
            .whenMatchedUpdate(
                set={
                    "is_current": "false",
                    "effective_to_date": f"date_sub(to_date('{batch_id}'), 1)",
                    "gold_load_timestamp": "current_timestamp()"
                }
            )
            .execute()
        )

        (
            changed_or_new_school_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(full_table_name)
        )

    print(f"Changed or new school rows inserted: {changed_or_new_count}")

else:
    raise ValueError(f"Unsupported run_mode: {run_mode}")

display(
    spark.table(full_table_name)
    .orderBy("school_id", "effective_from_date")
)

Changed or new school rows inserted: 2


school_scd_key,school_key,school_id,school_name,region,school_type,status,is_active_school,open_date,school_attribute_hash,effective_from_batch_id,effective_from_date,effective_to_date,is_current,bronze_record_id,source_file_name,run_id,silver_load_timestamp,gold_load_timestamp
2b119d4e079f26814136c3e6428caadc66e35a92c576a7ce171d69276cedfeb3,ab1e2e12fa3c39452ae34ede3becbaef0e3dfca111d25b31c248ad8fed7c3e8a,SCH001,ACT Education School 001,North Canberra,High School,Active,true,1994-09-04,3eec31597bdd7821cab7cba7c21931ee101dff1667bd8fbe92656a3aea313c20,2025-01-15,2025-01-15,null,true,9fa03a5f34e48abcbc3c36c3a3ea1e70f4e4e7a790f68314b4056bdcf94349dc,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,dev_education_qa_pipeline_2025-01-15_initial_job_333696814434230,2026-06-07T10:51:57.714Z,2026-06-08T03:56:56.120Z
a8080be5d47f5d69612d49349e2dc5900ee8fda68146032eb47a812eaa1bb725,9ce2d1f98bec04fa7e5edfc2cdf3a27e9046419e1a6c0b9900295edf65c7aa84,SCH002,ACT Education School 002,North Canberra,Primary,Active,true,2018-12-05,2b2f54e137fe88376446baf0cb15fa8ce938236749e4a5ea660024badc177913,2025-01-15,2025-01-15,null,true,c178b1273026d0dca9064fff44fbde8828eec70086f29426c2e9c49245aaf013,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,dev_education_qa_pipeline_2025-01-15_initial_job_333696814434230,2026-06-07T10:51:57.714Z,2026-06-08T03:56:56.120Z
35a559ca8e28b14848493a3953ee454ffce5681f317877eaf45fe8a72994a5a1,a30a149da02ad79f3d344894033c8c35edae78e9673679c4071b0468c204ccaf,SCH003,ACT Education School 003,North Canberra,Primary,Active,true,1978-05-29,1d98c3a03e3675dac7bd5e4135cc64bed94f20cb7bb525d4b5475fb7e736f963,2025-01-15,2025-01-15,null,true,e0278f9d95ecc1e20a7b8a6acbe17ebd55c9b2cb9b18c0dd82a8134edad07672,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,dev_education_qa_pipeline_2025-01-15_initial_job_333696814434230,2026-06-07T10:51:57.714Z,2026-06-08T03:56:56.120Z
29631e4e2787a85ee5d8b902b2e0d564909a3d347aa9d991a6d0b2209da1e3b0,e9f84eb2c537f7bdcf36a8278dcd71b5f406b0786f8f5172d778bff250ad4a7b,SCH004,ACT Education School 004,North Canberra,Primary,Active,true,1987-11-03,8594321ce049a15d2d877c4b98b38800da067218f92b4955eb52cf43e088e45c,2025-01-15,2025-01-15,null,true,75cbfa362a5b4e87a0edc496b60b6d117cd5ae650e5b01a48923b0d2031d4030,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,dev_education_qa_pipeline_2025-01-15_initial_job_333696814434230,2026-06-07T10:51:57.714Z,2026-06-08T03:56:56.120Z
16346ed3a5ca10f77ee76c85e5515bd5cd6c491b2698024c8ffb67c6f562b69d,4dc3fb611803262a20b19d71d8b7cfd92671a00206e44cf02321003392d92877,SCH005,ACT Education School 005,Tuggeranong,High School,Active,true,1989-10-11,d3ec32499b627f94b205b462da9ce7d56264862ab7cdc4a6aa08a876a3fc7f92,2025-01-15,2025-01-15,null,true,fe62286c15f3aba002a7017f55ced727b6c34b2df1f08db641091dde116d033d,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,dev_education_qa_pipeline_2025-01-15_initial_job_333696814434230,2026-06-07T10:51:57.714Z,2026-06-08T03:56:56.120Z
2050e10f85ec44e59406ff2da8102c8a1aae955d31e70ec781bb21013f5c3bba,068bfc9f9fd7eaf8f3a6babed6ef266c63d0b756a8fbd86b91c4450aa7c44922,SCH006,ACT Education School 006,Weston Creek,Primary,Active,true,1970-08-01,01ef0692edd5040b696c130bb5b4fdb99af89e8f27e47e5b089857a5bb96a059,2025-01-15,2025-01-15,null,true,77d4df079641cf679feed3348b4d308b6feab990fe49aed1003188780395cb29,abfss://education-data-lake@steduqadblakehouse.dfs.core.windows.net/raw/schools/batch_id=2025-01-15/schools.csv,dev_education_qa_pipeline_2025-01-15_initial_job_333696814434230,2026-06-07T10:51:57.714Z,2026-06-08T03:56:56.120Z
2563ba02568dc7e2f170870b537582686d51449b9a9a2eb7f83057408673c0fc,0542a656a5ad227ba6695b5973b4d826f2bd1df8efb3d6d2db2e2da5ef21c9cb,SCH007

In [0]:
display(
    spark.table(f"{catalog}.gold.dim_school")
    .agg(
        F.count("*").alias("school_rows"),
        F.countDistinct("school_key").alias("distinct_school_keys"),
        F.countDistinct("school_id").alias("distinct_school_ids")
    )
)

school_rows,distinct_school_keys,distinct_school_ids
52,51,51


In [0]:
display(
    spark.table(f"{catalog}.gold.dim_school")
    .groupBy("status")
    .count()
    .orderBy("status")
)

status,count
Active,49
Closed,3


In [0]:
# gold.dim_year_level
# Grain: one row per year level.

year_level_df = (
    spark.table(f"{catalog}.silver.students")
    .filter(F.col("batch_id") == batch_id)
    .select("year_level")
    .where(F.col("year_level").isNotNull())
    .distinct()
)

dim_year_level_df = (
    year_level_df
    .withColumn(
        "year_level_key",
        F.sha2(F.col("year_level").cast("string"), 256)
    )
    .withColumn(
        "year_level_label",
        F.when(F.col("year_level") == 0, F.lit("Kindergarten"))
        .otherwise(F.concat(F.lit("Year "), F.col("year_level").cast("string")))
    )
    .withColumn(
        "year_level_sort_order",
        F.col("year_level")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "year_level_key",
        "year_level",
        "year_level_label",
        "year_level_sort_order",
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/dim_year_level"

merge_condition = """
target.year_level_key = source.year_level_key
"""

write_delta_table(
    df=dim_year_level_df,
    table_name="gold.dim_year_level",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.dim_year_level")
    .orderBy("year_level_sort_order")
)

year_level_key,year_level,year_level_label,year_level_sort_order,gold_load_timestamp
5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9,0,Kindergarten,0,2026-06-08T05:16:15.444Z
6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b,1,Year 1,1,2026-06-08T05:16:15.444Z
d4735e3a265e16eee03f59718b9b5d03019c07d8b6c51f90da3a666eec13ab35,2,Year 2,2,2026-06-08T05:16:15.444Z
4e07408562bedb8b60ce05c1decfe3ad16b72230967de01f640b7e4729b49fce,3,Year 3,3,2026-06-08T05:16:15.444Z
4b227777d4dd1fc61c6f884f48641d02b4d121d3fd328cb08b5531fcacdabf8a,4,Year 4,4,2026-06-08T05:16:15.444Z
ef2d127de37b942baad06145e54b0c619a1f22327b2ebbcfbec78f5564afe39d,5,Year 5,5,2026-06-08T05:16:15.444Z
e7f6c011776e8db7cd330b54174fd76f7d0216b612387a5ffcfb81e6f0919683,6,Year 6,6,2026-06-08T05:16:15.444Z
7902699be42c8a8e46fbbb4501726517e86b22c56a189f7625a6da49081b2451,7,Year 7,7,2026-06-08T05:16:15.444Z
2c624232cdd221771294dfbb310aca000a0df6ac8b66b696d90ef06fdefb64a3,8,Year 8,8,2026-06-08T05:16:15.444Z
19581e27de7ced00ff1ce50b2047e7a567c76b1cbaebabe5ef03f7c3017bb5b7,9,Year 9,9,2026-06-08T05:16:15.444Z


In [0]:
# gold.dim_student
# Grain: one row per student per batch.
# Production note: student_id is synthetic in this portfolio.
# In production, hide/restrict raw student_id and expose student_key/student_batch_key.

students_df = (
    spark.table(f"{catalog}.silver.students")
    .filter(F.col("batch_id") == batch_id)
)

school_version_df = (
    spark.table(f"{catalog}.gold.dim_school")
    .select(
        F.col("school_key").alias("dim_school_key"),
        "school_scd_key",
        "effective_from_date",
        "effective_to_date"
    )
)

dim_student_df = (
    students_df
    .select(
        "batch_id",
        "student_id",
        "school_id",
        "year_level",
        "gender",
        "enrolment_date",
        "status",
        "bronze_record_id",
        "source_file_name",
        "run_id",
        "silver_load_timestamp"
    )
    .withColumn("batch_date", F.to_date(F.lit(batch_id)))
    .withColumn("student_key", F.sha2(F.coalesce(F.col("student_id"), F.lit("")), 256))
    .withColumn(
        "student_batch_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("batch_id"), F.lit("")),
                F.coalesce(F.col("student_id"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("school_key", F.sha2(F.coalesce(F.col("school_id"), F.lit("")), 256))
    .withColumn("year_level_key", F.sha2(F.coalesce(F.col("year_level").cast("string"), F.lit("")), 256))
    .join(
        school_version_df,
        (F.col("school_key") == F.col("dim_school_key"))
        & (F.col("batch_date") >= F.col("effective_from_date"))
        & (
            F.col("effective_to_date").isNull()
            | (F.col("batch_date") <= F.col("effective_to_date"))
        ),
        "left"
    )
    .withColumn(
        "is_active_student",
        F.when(F.col("status") == "Active", F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "student_batch_key",
        "student_key",
        "batch_id",
        "student_id",
        "school_key",
        "school_scd_key",
        "school_id",
        "year_level_key",
        "year_level",
        "gender",
        "enrolment_date",
        "status",
        "is_active_student",
        "bronze_record_id",
        "source_file_name",
        "run_id",
        "silver_load_timestamp",
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/dim_student"

merge_condition = """
target.student_batch_key = source.student_batch_key
"""

write_delta_table(
    df=dim_student_df,
    table_name="gold.dim_student",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.dim_student")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("student_batch_rows"),
        F.countDistinct("student_key").alias("distinct_students"),
        F.countDistinct("student_batch_key").alias("distinct_student_batch_keys"),
        F.sum(F.when(F.col("school_scd_key").isNull(), 1).otherwise(0)).alias("missing_school_scd_key")
    )
    .orderBy("batch_id")
)

batch_id,student_batch_rows,distinct_students,distinct_student_batch_keys,missing_school_scd_key
2025-01-15,10002,10002,10002,1
2026-01-15,10502,10502,10502,1


In [0]:
# gold.dim_assessment_domain
# Grain: one row per assessment domain.

domain_df = (
    spark.table(f"{catalog}.silver.assessment_results")
    .filter(F.col("batch_id") == batch_id)
    .select("domain")
    .where(F.col("domain").isNotNull())
    .distinct()
)

dim_assessment_domain_df = (
    domain_df
    .withColumn(
        "domain_key",
        F.sha2(F.coalesce(F.col("domain"), F.lit("")), 256)
    )
    .withColumn(
        "domain_sort_order",
        F.when(F.col("domain") == "Reading", F.lit(1))
        .when(F.col("domain") == "Writing", F.lit(2))
        .when(F.col("domain") == "Numeracy", F.lit(3))
        .otherwise(F.lit(99))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "domain_key",
        "domain",
        "domain_sort_order",
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/dim_assessment_domain"

merge_condition = """
target.domain_key = source.domain_key
"""

write_delta_table(
    df=dim_assessment_domain_df,
    table_name="gold.dim_assessment_domain",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.dim_assessment_domain")
    .orderBy("domain_sort_order")
)

domain_key,domain,domain_sort_order,gold_load_timestamp
463816d07097e1a7cf27b86cf03082ee49f6ce2d87663cd27a24d7254a83e76a,Reading,1,2026-06-08T05:29:45.576Z
a8bfae3eee941527f2568d7e1ae4d526cc1c764fd09ee1e62deb13e5f00c6078,Writing,2,2026-06-08T05:29:45.576Z
17d13d6bf65379799de0763b9ab6eee6020faf0e20ddb4b52ae14b4b1c9c9da4,Numeracy,3,2026-06-08T05:29:45.576Z


In [0]:
# COMMAND ----------

# gold.dim_proficiency_band
# Grain: one row per proficiency band.

proficiency_band_df = (
    spark.table(f"{catalog}.silver.assessment_results")
    .filter(F.col("batch_id") == batch_id)
    .select("proficiency_band")
    .where(F.col("proficiency_band").isNotNull())
    .distinct()
)

dim_proficiency_band_df = (
    proficiency_band_df
    .withColumn(
        "proficiency_band_key",
        F.sha2(F.coalesce(F.col("proficiency_band"), F.lit("")), 256)
    )
    .withColumn(
        "proficiency_band_sort_order",
        F.when(F.col("proficiency_band") == "Low", F.lit(1))
        .when(F.col("proficiency_band") == "Medium", F.lit(2))
        .when(F.col("proficiency_band") == "High", F.lit(3))
        .otherwise(F.lit(99))
    )
    .withColumn(
        "score_range_label",
        F.when(F.col("proficiency_band") == "Low", F.lit("250 to 399"))
        .when(F.col("proficiency_band") == "Medium", F.lit("400 to 549"))
        .when(F.col("proficiency_band") == "High", F.lit("550 and above"))
        .otherwise(F.lit("Unknown"))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "proficiency_band_key",
        "proficiency_band",
        "proficiency_band_sort_order",
        "score_range_label",
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/dim_proficiency_band"

merge_condition = """
target.proficiency_band_key = source.proficiency_band_key
"""

write_delta_table(
    df=dim_proficiency_band_df,
    table_name="gold.dim_proficiency_band",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.dim_proficiency_band")
    .orderBy("proficiency_band_sort_order")
)

proficiency_band_key,proficiency_band,proficiency_band_sort_order,score_range_label,gold_load_timestamp
f793de205ead5ac302c4a1627829dea41f176b1068b993a32373fc869918374b,Low,1,250 to 399,2026-06-08T05:29:49.696Z
8e588cd187741f1cd76f5fab77b7208782a8c21d764ce7d7a4cf3ac4e0968873,Medium,2,400 to 549,2026-06-08T05:29:49.696Z
c4ebc6d4a5832cd9415f906ad03661110c705a72381c8b8b145761d02e2dd23a,High,3,550 and above,2026-06-08T05:29:49.696Z


In [0]:
# gold.dim_dq_rule
# Grain: one row per data quality rule.

dq_rule_df = spark.table(f"{catalog}.qa.dq_rule_catalog")

dim_dq_rule_df = (
    dq_rule_df
    .select(
        "rule_id",
        "rule_name",
        "target_table",
        "severity",
        "business_rule",
        "expected_outcome"
    )
    .withColumn(
        "dq_rule_key",
        F.sha2(F.coalesce(F.col("rule_id"), F.lit("")), 256)
    )
    .withColumn(
        "severity_sort_order",
        F.when(F.col("severity") == "High", F.lit(1))
         .when(F.col("severity") == "Medium", F.lit(2))
         .when(F.col("severity") == "Low", F.lit(3))
         .otherwise(F.lit(99))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "dq_rule_key",
        "rule_id",
        "rule_name",
        "target_table",
        "severity",
        "severity_sort_order",
        "business_rule",
        "expected_outcome",
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/dim_dq_rule"

merge_condition = """
target.dq_rule_key = source.dq_rule_key
"""

write_delta_table(
    df=dim_dq_rule_df,
    table_name="gold.dim_dq_rule",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.dim_dq_rule")
    .orderBy("rule_id")
)

dq_rule_key,rule_id,rule_name,target_table,severity,severity_sort_order,business_rule,expected_outcome,gold_load_timestamp
82c309478f037a1ead58fd76122bae3a1042ce3fa2a391f44104b40d88168580,DQ001,Missing student ID,silver.students,High,1,student_id must not be null or blank,No missing student IDs,2026-06-08T05:29:54.914Z
6a104013abe0243a69e01750c02b251ee0818190e8ff2d13a3cfd6794887b9cc,DQ002,Missing school ID,silver.students,High,1,school_id must not be null or blank,No missing school IDs,2026-06-08T05:29:54.914Z
f3c6ec02c05d5775a3818e6a42e58884b01930cb357e3b14add4bb8c52fafbe1,DQ003,Invalid attendance days,silver.attendance,High,1,attended_days must be between 0 and possible_days,Attendance days are within valid range,2026-06-08T05:29:54.914Z
fb966fa3660c03fdf4614719372a90944f2b82c1b9e037f0cf8b44c16f297695,DQ004,Duplicate attendance business record,silver.attendance,Medium,2,"student_id, school_id, attendance_month should be unique",No duplicate attendance records,2026-06-08T05:29:54.914Z
9b2eacbe54a52f0a6bd1086b109fff8282fa2165ac28d0f4720ca5bb74c9dd17,DQ005,Attendance references missing student,silver.attendance,High,1,attendance.student_id must exist in silver.students,No orphan attendance records,2026-06-08T05:29:54.914Z
bbfc8c9ee5e7c9a94a217be053b9632e6d9f6ad1a744a328a5a79f5d3e4d24c3,DQ006,Assessment references missing student,silver.assessment_results,High,1,assessment_results.student_id must exist in silver.students,No orphan assessment records,2026-06-08T05:29:54.914Z
e1eb11d5cbae2d953b6784ecd5fbbaf206149788438c1522d8ac59e02d8d57b1,DQ007,Invalid assessment score,silver.assessment_results,High,1,score must be between 250 and 700,Assessment scores are within the valid 250 to 700 scale,2026-06-08T05:29:54.914Z
cf0f5138edab5d3466165e8b03529d9f398fe91a7792c20eaa59b960a5a737a1,DQ008,Invalid proficiency band,silver.assessment_results,Medium,2,"proficiency_band must be one of Low, Medium, High",Proficiency bands are valid,2026-06-08T05:29:54.914Z
9b79466c33f877e41302fb9015a26b067f337d340c966336a09be1b066e273a7,DQ009,Invalid school status,silver.schools,Medium,2,status must be Active or Closed,School status values are valid,2026-06-08T05:29:54.914Z
d780e4e60c31f61a4120a19bed2a426535f4aec6956c7ed80e4bc1434f8e3339,DQ010,Future attendance month,silver.attendance,Medium,2,attendance_month must not be in the future,No future attendance month,2026-06-08T05:29:54.914Z


### gold.fact_attendance, gold.fact_assessment_result, gold.fact_data_quality_result, gold.fact_defect

In [0]:
# gold.fact_attendance
# Grain: one row per valid attendance record.
# Excludes attendance records failed by DQ003, DQ004, or DQ005.

latest_qa_run_row = (
    spark.table(f"{catalog}.qa.dq_validation_results")
    .filter(F.col("batch_id") == batch_id)
    .orderBy(F.col("run_timestamp").desc())
    .select("run_id")
    .first()
)

if latest_qa_run_row is None:
    raise ValueError(f"No QA run found for batch_id={batch_id}")

latest_qa_run_id = latest_qa_run_row["run_id"]

attendance_df = (
    spark.table(f"{catalog}.silver.attendance")
    .filter(F.col("batch_id") == batch_id)
)

dim_student_keys_df = (
    spark.table(f"{catalog}.gold.dim_student")
    .filter(F.col("batch_id") == batch_id)
    .select(
        "batch_id",
        "student_id",
        "student_key",
        "student_batch_key",
        "year_level_key"
    )
)

dim_school_versions_df = (
    spark.table(f"{catalog}.gold.dim_school")
    .select(
        F.col("school_key").alias("dim_school_key"),
        "school_scd_key",
        "effective_from_date",
        "effective_to_date"
    )
)

invalid_attendance_records_df = (
    spark.table(f"{catalog}.qa.dq_failed_records")
    .filter(
        (F.col("run_id") == latest_qa_run_id)
        & (F.col("batch_id") == batch_id)
        & (F.col("target_table") == "silver.attendance")
        & (F.col("rule_id").isin(["DQ003", "DQ004", "DQ005"]))
    )
    .select("bronze_record_id")
    .distinct()
)

valid_attendance_df = (
    attendance_df
    .join(
        invalid_attendance_records_df,
        on="bronze_record_id",
        how="left_anti"
    )
)

fact_attendance_df = (
    valid_attendance_df.alias("a")
    .withColumn("batch_date", F.to_date(F.lit(batch_id)))
    .withColumn("school_key", F.sha2(F.coalesce(F.col("a.school_id"), F.lit("")), 256))
    .join(
        dim_student_keys_df.alias("stu"),
        on=["batch_id", "student_id"],
        how="left"
    )
    .join(
        dim_school_versions_df.alias("sch"),
        (F.col("school_key") == F.col("sch.dim_school_key"))
        & (F.col("batch_date") >= F.col("sch.effective_from_date"))
        & (
            F.col("sch.effective_to_date").isNull()
            | (F.col("batch_date") <= F.col("sch.effective_to_date"))
        ),
        how="left"
    )
    .withColumn(
        "attendance_fact_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("a.batch_id"), F.lit("")),
                F.coalesce(F.col("a.attendance_id"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("batch_key", F.sha2(F.coalesce(F.col("a.batch_id"), F.lit("")), 256))
    .withColumn(
        "attendance_month_date_key",
        F.date_format(F.col("a.attendance_month"), "yyyyMMdd").cast("int")
    )
    .withColumn(
        "attendance_rate",
        F.when(
            F.col("a.possible_days") > 0,
            F.round(F.col("a.attended_days") / F.col("a.possible_days"), 4)
        ).otherwise(F.lit(None))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "attendance_fact_key",
        "batch_key",
        "school_key",
        "school_scd_key",
        "student_key",
        "student_batch_key",
        "year_level_key",
        "attendance_month_date_key",
        F.col("a.batch_id").alias("batch_id"),
        F.col("a.attendance_id").alias("attendance_id"),
        F.col("a.student_id").alias("student_id"),
        F.col("a.school_id").alias("school_id"),
        F.col("a.attendance_month").alias("attendance_month"),
        F.col("a.possible_days").alias("possible_days"),
        F.col("a.attended_days").alias("attended_days"),
        "attendance_rate",
        F.col("a.absence_reason").alias("absence_reason"),
        F.col("a.bronze_record_id").alias("bronze_record_id"),
        F.col("a.run_id").alias("source_run_id"),
        F.col("a.silver_load_timestamp").alias("silver_load_timestamp"),
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/fact_attendance"

merge_condition = """
target.attendance_fact_key = source.attendance_fact_key
"""

write_delta_table(
    df=fact_attendance_df,
    table_name="gold.fact_attendance",
    target_path=target_path,
    merge_condition=merge_condition
)

In [0]:
display(
    spark.table(f"{catalog}.gold.fact_attendance")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("fact_attendance_rows"),
        F.countDistinct("attendance_fact_key").alias("distinct_fact_keys"),
        F.sum(F.when(F.col("school_scd_key").isNull(), 1).otherwise(0)).alias("missing_school_scd_key"),
        F.sum(F.when(F.col("student_batch_key").isNull(), 1).otherwise(0)).alias("missing_student_batch_key"),
        F.min("attendance_rate").alias("min_attendance_rate"),
        F.max("attendance_rate").alias("max_attendance_rate")
    )
    .orderBy("batch_id")
)

batch_id,fact_attendance_rows,distinct_fact_keys,missing_school_scd_key,missing_student_batch_key,min_attendance_rate,max_attendance_rate
2025-01-15,119998,119998,0,0,0.4,1.0
2026-01-15,96934,96934,0,0,0.4,1.0


In [0]:
# gold.fact_assessment_result
# Grain: one row per valid assessment result.
# Excludes assessment records failed by DQ006, DQ007, or DQ008.

latest_qa_run_row = (
    spark.table(f"{catalog}.qa.dq_validation_results")
    .filter(F.col("batch_id") == batch_id)
    .orderBy(F.col("run_timestamp").desc())
    .select("run_id")
    .first()
)

if latest_qa_run_row is None:
    raise ValueError(f"No QA run found for batch_id={batch_id}")

latest_qa_run_id = latest_qa_run_row["run_id"]

assessment_df = (
    spark.table(f"{catalog}.silver.assessment_results")
    .filter(F.col("batch_id") == batch_id)
)

dim_student_keys_df = (
    spark.table(f"{catalog}.gold.dim_student")
    .filter(F.col("batch_id") == batch_id)
    .select(
        "batch_id",
        "student_id",
        "student_key",
        "student_batch_key",
        "school_key",
        "school_scd_key"
    )
)

dim_domain_df = (
    spark.table(f"{catalog}.gold.dim_assessment_domain")
    .select(
        "domain",
        "domain_key"
    )
)

dim_proficiency_band_df = (
    spark.table(f"{catalog}.gold.dim_proficiency_band")
    .select(
        "proficiency_band",
        "proficiency_band_key"
    )
)

invalid_assessment_records_df = (
    spark.table(f"{catalog}.qa.dq_failed_records")
    .filter(
        (F.col("run_id") == latest_qa_run_id)
        & (F.col("batch_id") == batch_id)
        & (F.col("target_table") == "silver.assessment_results")
        & (F.col("rule_id").isin(["DQ006", "DQ007", "DQ008"]))
    )
    .select("bronze_record_id")
    .distinct()
)

valid_assessment_df = (
    assessment_df
    .join(
        invalid_assessment_records_df,
        on="bronze_record_id",
        how="left_anti"
    )
)

fact_assessment_result_df = (
    valid_assessment_df.alias("a")
    .join(
        dim_student_keys_df.alias("stu"),
        on=["batch_id", "student_id"],
        how="left"
    )
    .join(
        dim_domain_df.alias("dom"),
        F.col("a.domain") == F.col("dom.domain"),
        how="left"
    )
    .join(
        dim_proficiency_band_df.alias("band"),
        F.col("a.proficiency_band") == F.col("band.proficiency_band"),
        how="left"
    )
    .withColumn(
        "assessment_fact_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("a.batch_id"), F.lit("")),
                F.coalesce(F.col("a.assessment_id"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("batch_key", F.sha2(F.coalesce(F.col("a.batch_id"), F.lit("")), 256))
    .withColumn(
        "assessment_year_date_key",
        F.date_format(
            F.to_date(F.concat(F.col("a.assessment_year").cast("string"), F.lit("-01-01"))),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "assessment_fact_key",
        "batch_key",
        "school_key",
        "school_scd_key",
        "student_key",
        "student_batch_key",
        "domain_key",
        "proficiency_band_key",
        "assessment_year_date_key",
        F.col("a.batch_id").alias("batch_id"),
        F.col("a.assessment_id").alias("assessment_id"),
        F.col("a.student_id").alias("student_id"),
        F.col("a.school_id").alias("school_id"),
        F.col("a.assessment_year").alias("assessment_year"),
        F.col("a.domain").alias("domain"),
        F.col("a.score").alias("score"),
        F.col("a.proficiency_band").alias("proficiency_band"),
        F.col("a.bronze_record_id").alias("bronze_record_id"),
        F.col("a.run_id").alias("source_run_id"),
        F.col("a.silver_load_timestamp").alias("silver_load_timestamp"),
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/fact_assessment_result"

merge_condition = """
target.assessment_fact_key = source.assessment_fact_key
"""

write_delta_table(
    df=fact_assessment_result_df,
    table_name="gold.fact_assessment_result",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.fact_assessment_result")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("fact_assessment_rows"),
        F.countDistinct("assessment_fact_key").alias("distinct_fact_keys"),
        F.sum(F.when(F.col("school_scd_key").isNull(), 1).otherwise(0)).alias("missing_school_scd_key"),
        F.sum(F.when(F.col("student_batch_key").isNull(), 1).otherwise(0)).alias("missing_student_batch_key"),
        F.sum(F.when(F.col("domain_key").isNull(), 1).otherwise(0)).alias("missing_domain_key"),
        F.sum(F.when(F.col("proficiency_band_key").isNull(), 1).otherwise(0)).alias("missing_proficiency_band_key"),
        F.min("score").alias("min_score"),
        F.max("score").alias("max_score")
    )
    .orderBy("batch_id")
)

batch_id,fact_assessment_rows,distinct_fact_keys,missing_school_scd_key,missing_student_batch_key,missing_domain_key,missing_proficiency_band_key,min_score,max_score
2025-01-15,30000,30000,0,0,0,0,250,670
2026-01-15,24234,24234,0,0,0,0,250,670


In [0]:
# gold.fact_data_quality_result
# Grain: one row per data quality rule result per batch for the latest QA run.

latest_qa_run_row = (
    spark.table(f"{catalog}.qa.dq_validation_results")
    .filter(F.col("batch_id") == batch_id)
    .orderBy(F.col("run_timestamp").desc())
    .select("run_id")
    .first()
)

if latest_qa_run_row is None:
    raise ValueError(f"No QA run found for batch_id={batch_id}")

latest_qa_run_id = latest_qa_run_row["run_id"]

dq_results_df = (
    spark.table(f"{catalog}.qa.dq_validation_results")
    .filter(
        (F.col("batch_id") == batch_id)
        & (F.col("run_id") == latest_qa_run_id)
    )
)

dim_dq_rule_df = (
    spark.table(f"{catalog}.gold.dim_dq_rule")
    .select(
        "rule_id",
        "dq_rule_key",
        "severity_sort_order"
    )
)

fact_data_quality_result_df = (
    dq_results_df.alias("dq")
    .join(
        dim_dq_rule_df.alias("rule"),
        on="rule_id",
        how="left"
    )
    .withColumn(
        "dq_result_fact_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("dq.run_id"), F.lit("")),
                F.coalesce(F.col("dq.batch_id"), F.lit("")),
                F.coalesce(F.col("dq.rule_id"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("batch_key", F.sha2(F.coalesce(F.col("dq.batch_id"), F.lit("")), 256))
    .withColumn(
        "status_sort_order",
        F.when(F.col("dq.status") == "FAIL", F.lit(1))
        .when(F.col("dq.status") == "WARN", F.lit(2))
        .when(F.col("dq.status") == "PASS", F.lit(3))
        .otherwise(F.lit(9))
    )
    .withColumn(
        "issue_flag",
        F.when(F.col("dq.status").isin("FAIL", "WARN"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "dq_result_fact_key",
        "batch_key",
        "dq_rule_key",
        F.col("dq.run_id").alias("run_id"),
        F.col("dq.batch_id").alias("batch_id"),
        F.col("dq.rule_id").alias("rule_id"),
        F.col("dq.rule_name").alias("rule_name"),
        F.col("dq.target_table").alias("target_table"),
        F.col("dq.severity").alias("severity"),
        "severity_sort_order",
        F.col("dq.status").alias("status"),
        "status_sort_order",
        F.col("dq.failed_record_count").alias("failed_record_count"),
        "issue_flag",
        F.col("dq.run_timestamp").alias("run_timestamp"),
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/fact_data_quality_result"

merge_condition = """
target.dq_result_fact_key = source.dq_result_fact_key
"""

write_delta_table(
    df=fact_data_quality_result_df,
    table_name="gold.fact_data_quality_result",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.fact_data_quality_result")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("dq_result_rows"),
        F.countDistinct("dq_result_fact_key").alias("distinct_fact_keys"),
        F.sum("failed_record_count").alias("failed_record_count"),
        F.sum("issue_flag").alias("issue_rule_results"),
        F.sum(F.when(F.col("dq_rule_key").isNull(), 1).otherwise(0)).alias("missing_dq_rule_key")
    )
    .orderBy("batch_id")
)

batch_id,dq_result_rows,distinct_fact_keys,failed_record_count,issue_rule_results,missing_dq_rule_key
2025-01-15,11,11,14,7,0
2026-01-15,11,11,8,5,0


In [0]:
# gold.fact_defect
# Grain: one row per defect per batch for the latest QA run.

latest_qa_run_row = (
    spark.table(f"{catalog}.qa.dq_validation_results")
    .filter(F.col("batch_id") == batch_id)
    .orderBy(F.col("run_timestamp").desc())
    .select("run_id")
    .first()
)

if latest_qa_run_row is None:
    raise ValueError(f"No QA run found for batch_id={batch_id}")

latest_qa_run_id = latest_qa_run_row["run_id"]

defect_df = (
    spark.table(f"{catalog}.qa.defect_log")
    .filter(
        (F.col("batch_id") == batch_id)
        & (F.col("run_id") == latest_qa_run_id)
    )
)

dim_dq_rule_df = (
    spark.table(f"{catalog}.gold.dim_dq_rule")
    .select(
        "rule_id",
        "dq_rule_key",
        "rule_name",
        "target_table",
        "severity_sort_order"
    )
)

fact_defect_df = (
    defect_df.alias("d")
    .join(
        dim_dq_rule_df.alias("rule"),
        on="rule_id",
        how="left"
    )
    .withColumn(
        "defect_fact_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("d.defect_id"), F.lit("")),
                F.coalesce(F.col("d.run_id"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("batch_key", F.sha2(F.coalesce(F.col("d.batch_id"), F.lit("")), 256))
    .withColumn(
        "defect_status_sort_order",
        F.when(F.col("d.defect_status") == "Open", F.lit(1))
        .when(F.col("d.defect_status") == "In Progress", F.lit(2))
        .when(F.col("d.defect_status") == "Resolved", F.lit(3))
        .when(F.col("d.defect_status") == "Closed", F.lit(4))
        .otherwise(F.lit(9))
    )
    .withColumn(
        "open_defect_flag",
        F.when(F.col("d.defect_status") == "Open", F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
    .select(
        "defect_fact_key",
        "batch_key",
        "dq_rule_key",
        F.col("d.defect_id").alias("defect_id"),
        F.col("d.run_id").alias("run_id"),
        F.col("d.batch_id").alias("batch_id"),
        F.col("d.rule_id").alias("rule_id"),
        F.col("rule.rule_name").alias("rule_name"),
        F.col("rule.target_table").alias("target_table"),
        F.col("d.severity").alias("severity"),
        "severity_sort_order",
        F.col("d.defect_title").alias("defect_title"),
        F.col("d.defect_status").alias("defect_status"),
        "defect_status_sort_order",
        "open_defect_flag",
        F.col("d.failed_record_count").alias("failed_record_count"),
        F.col("d.recommended_action").alias("recommended_action"),
        F.col("d.created_timestamp").alias("created_timestamp"),
        "gold_load_timestamp"
    )
)

target_path = f"{lake_root}/gold/fact_defect"

merge_condition = """
target.defect_fact_key = source.defect_fact_key
"""

write_delta_table(
    df=fact_defect_df,
    table_name="gold.fact_defect",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.fact_defect")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("defect_rows"),
        F.countDistinct("defect_fact_key").alias("distinct_fact_keys"),
        F.sum("failed_record_count").alias("defect_failed_record_count"),
        F.sum("open_defect_flag").alias("open_defects"),
        F.sum(F.when(F.col("dq_rule_key").isNull(), 1).otherwise(0)).alias("missing_dq_rule_key")
    )
    .orderBy("batch_id")
)

batch_id,defect_rows,distinct_fact_keys,defect_failed_record_count,open_defects,missing_dq_rule_key
2025-01-15,7,7,14,7,0
2026-01-15,5,5,8,5,0


### gold.data_quality_summary, gold.data_quality_rule_detail

In [0]:
dq_results_df = (
    spark.table(f"{catalog}.qa.dq_validation_results")
    .filter(F.col("run_id") == run_id)
)

data_quality_summary_df = (
    dq_results_df
    .groupBy(
        "run_id",
        "batch_id",
        "severity",
        "status",
    )
    .agg(
        F.countDistinct("rule_id").alias("rule_count"),
        F.sum("failed_record_count").alias("failed_record_count")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

target_path = f"{lake_root}/gold/data_quality_summary"


merge_condition = """
target.run_id = source.run_id
AND target.batch_id = source.batch_id
AND target.severity = source.severity
AND target.status = source.status
"""

write_delta_table(
    data_quality_summary_df, 
    "gold.data_quality_summary", 
    target_path,
    merge_condition
)

display(
    spark.table(f"{catalog}.gold.data_quality_summary")
    .orderBy("batch_id", "severity", "status")
)

run_id,batch_id,severity,status,rule_count,failed_record_count,gold_load_timestamp
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,High,FAIL,5,5,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,High,PASS,1,0,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,Low,WARN,1,5,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,Medium,FAIL,1,4,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,Medium,PASS,3,0,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,High,FAIL,4,4,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,High,PASS,2,0,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,Low,PASS,1,0,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,Medium,FAIL,1,4,2026-06-07T01:29:54.682Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,Medium,PASS,3,0,2026-06-07T01:29:54.682Z


In [0]:
# gold.data_quality_rule_detail

dq_results_df = (
    spark.table(f"{catalog}.qa.dq_validation_results")
    .filter(F.col("run_id") == run_id)
)

data_quality_rule_detail_df = (
    dq_results_df
    .select(
        "run_id",
        "batch_id",
        "rule_id",
        "rule_name",
        "target_table",
        "severity",
        "status",
        "failed_record_count",
        "run_timestamp"
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

target_path = f"{lake_root}/gold/data_quality_rule_detail"

merge_condition = """
target.run_id = source.run_id
AND target.batch_id = source.batch_id
AND target.rule_id = source.rule_id
"""

write_delta_table(
    df=data_quality_rule_detail_df,
    table_name="gold.data_quality_rule_detail",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.data_quality_rule_detail")
    .orderBy("rule_id", "batch_id")
)

run_id,batch_id,rule_id,rule_name,target_table,severity,status,failed_record_count,run_timestamp,gold_load_timestamp
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,DQ001,Missing student ID,silver.students,High,FAIL,1,2026-06-06T10:07:28.796Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,DQ001,Missing student ID,silver.students,High,PASS,0,2026-06-06T10:07:31.591Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,DQ002,Missing school ID,silver.students,High,FAIL,1,2026-06-06T10:09:41.396Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,DQ002,Missing school ID,silver.students,High,FAIL,1,2026-06-06T10:09:38.734Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,DQ003,Invalid attendance days,silver.attendance,High,FAIL,1,2026-06-06T10:10:09.544Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,DQ003,Invalid attendance days,silver.attendance,High,FAIL,1,2026-06-06T10:10:12.184Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,DQ004,Duplicate attendance business record,silver.attendance,Medium,FAIL,4,2026-06-06T10:10:38.040Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,DQ004,Duplicate attendance business record,silver.attendance,Medium,FAIL,4,2026-06-06T10:10:30.119Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2025-01-15,DQ005,Attendance references missing student,silver.attendance,High,FAIL,1,2026-06-06T10:10:50.748Z,2026-06-07T01:33:22.398Z
dev_education_qa_data_quality_checks_20260606T083052Z,2026-01-15,DQ005,Attendance references missing student,silver.attendance,High,FAIL,1,2026-06-06T10:10:53.605Z,2026-06-07T01:33:22.398Z


### gold.attendance_by_school_month, gold.attendance_by_year_level

In [0]:
# gold.attendance_by_school_month

attendance_df = (
    spark.table(f"{catalog}.silver.attendance")
    .filter(F.col("batch_id") == batch_id)
)
schools_df = (
    spark.table(f"{catalog}.silver.schools")
    .filter(F.col("batch_id") == batch_id)
)

invalid_attendance_records_df = (
    spark.table(f"{catalog}.qa.dq_failed_records")
    .filter(
        (F.col("run_id") == run_id)
        & (F.col("target_table") == "silver.attendance")
        & (F.col("rule_id").isin(["DQ003", "DQ004", "DQ005"]))
    )
    .select("bronze_record_id")
    .distinct()
)

valid_attendance_df = (
    attendance_df.alias("a")
    .join(
        invalid_attendance_records_df.alias("bad"),
        on="bronze_record_id",
        how="left_anti"
    )
)

attendance_by_school_month_df = (
    valid_attendance_df.alias("a")
    .join(
        schools_df.select(
            "batch_id",
            "school_id",
            "school_name",
            "region",
            "school_type",
            "status"
        ).alias("s"),
        on=["batch_id", "school_id"],
        how="left"
    )
    .groupBy(
        "batch_id",
        "school_id",
        "school_name",
        "region",
        "school_type",
        "attendance_month"
    )
    .agg(
        F.sum("possible_days").alias("possible_days"),
        F.sum("attended_days").alias("attended_days"),
        F.countDistinct("student_id").alias("student_count")
    )
    .withColumn(
        "attendance_rate",
        F.when(
            F.col("possible_days") > 0,
            F.round(F.col("attended_days") / F.col("possible_days"), 4)
        ).otherwise(F.lit(None))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)


target_path = f"{lake_root}/gold/attendance_by_school_month"

merge_condition = """
target.batch_id = source.batch_id
AND target.school_id = source.school_id
AND target.attendance_month = source.attendance_month
"""

write_delta_table(
    df=attendance_by_school_month_df,
    table_name="gold.attendance_by_school_month",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.attendance_by_school_month")
    .orderBy("batch_id", "school_id", "attendance_month")
    .limit(50)
)

batch_id,school_id,school_name,region,school_type,attendance_month,possible_days,attended_days,student_count,attendance_rate,gold_load_timestamp
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-01-01,0,0,200,null,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-02-01,3985,3418,200,0.8577,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-03-01,3975,3405,200,0.8566,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-04-01,4015,3387,200,0.8436,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-05-01,3991,3422,200,0.8574,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-06-01,3981,3340,200,0.839,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-07-01,4003,3347,200,0.8361,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-08-01,4030,3444,200,0.8546,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-09-01,3980,3429,200,0.8616,2026-06-07T01:49:38.768Z
2025-01-15,SCH001,ACT Education School 001,North Canberra,High School,2024-10-01,3988,3383,200,0.8483,2026-06-07T01:49:38.768Z


In [0]:
(
    spark.table(f"{catalog}.gold.attendance_by_school_month")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("school_month_rows"),
        F.min("attendance_rate").alias("min_attendance_rate"),
        F.max("attendance_rate").alias("max_attendance_rate")
    )
    .orderBy("batch_id")
    .show()
)

+----------+-----------------+-------------------+-------------------+
|  batch_id|school_month_rows|min_attendance_rate|max_attendance_rate|
+----------+-----------------+-------------------+-------------------+
|2025-01-15|              600|              0.736|             0.8717|
|2026-01-15|              576|             0.7284|             0.8951|
+----------+-----------------+-------------------+-------------------+



In [0]:
(
    valid_attendance_df
    .groupBy("batch_id")
    .agg(
        F.countDistinct("school_id").alias("schools_with_valid_attendance"),
        F.countDistinct("attendance_month").alias("attendance_months")
    )
    .orderBy("batch_id")
    .show()
)

+----------+-----------------------------+-----------------+
|  batch_id|schools_with_valid_attendance|attendance_months|
+----------+-----------------------------+-----------------+
|2025-01-15|                           50|               12|
|2026-01-15|                           48|               12|
+----------+-----------------------------+-----------------+



In [0]:
# gold.attendance_by_year_level

attendance_df = (
    spark.table(f"{catalog}.silver.attendance")
    .filter(F.col("batch_id") == batch_id)
)
students_df = (
    spark.table(f"{catalog}.silver.students")
    .filter(F.col("batch_id") == batch_id)
)

invalid_attendance_records_df = (
    spark.table(f"{catalog}.qa.dq_failed_records")
    .filter(
        (F.col("run_id") == run_id)
        & (F.col("target_table") == "silver.attendance")
        & (F.col("rule_id").isin(["DQ003", "DQ004", "DQ005"]))
    )
    .select("bronze_record_id")
    .distinct()
)

valid_attendance_df = (
    attendance_df
    .join(
        invalid_attendance_records_df,
        on="bronze_record_id",
        how="left_anti"
    )
)


attendance_by_year_level_df = (
    valid_attendance_df.alias("a")
    .join(
        students_df.select(
            "batch_id",
            "student_id",
            "year_level",
            "status"
        ).alias("s"),
        on=["batch_id", "student_id"],
        how="left"
    )
    .groupBy(
        "batch_id",
        "year_level",
        "attendance_month"
    )
    .agg(
        F.sum("possible_days").alias("possible_days"),
        F.sum("attended_days").alias("attended_days"),
        F.countDistinct("student_id").alias("student_count")
    )
    .withColumn(
        "attendance_rate",
        F.when(
            F.col("possible_days") > 0,
            F.round(F.col("attended_days") / F.col("possible_days"), 4)
        ).otherwise(F.lit(None))
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

target_path = f"{lake_root}/gold/attendance_by_year_level"

merge_condition = """
target.batch_id = source.batch_id
AND target.year_level = source.year_level
AND target.attendance_month = source.attendance_month
"""

write_delta_table(
    df=attendance_by_year_level_df,
    table_name="gold.attendance_by_year_level",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.attendance_by_year_level")
    .orderBy("batch_id", "year_level", "attendance_month")
    .limit(50)
)

batch_id,year_level,attendance_month,possible_days,attended_days,student_count,attendance_rate,gold_load_timestamp
2025-01-15,0,2024-01-01,0,0,971,null,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-02-01,19410,16505,971,0.8503,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-03-01,19458,16501,971,0.848,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-04-01,19475,16622,971,0.8535,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-05-01,19520,16601,971,0.8505,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-06-01,19414,16489,971,0.8493,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-07-01,19315,16503,971,0.8544,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-08-01,19462,16629,971,0.8544,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-09-01,19512,16609,971,0.8512,2026-06-07T02:14:13.625Z
2025-01-15,0,2024-10-01,19502,16642,971,0.8533,2026-06-07T02:14:13.625Z


In [0]:
(
    spark.table(f"{catalog}.gold.attendance_by_year_level")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("year_level_month_rows"),
        F.min("attendance_rate").alias("min_attendance_rate"),
        F.max("attendance_rate").alias("max_attendance_rate")
    )
    .orderBy("batch_id")
    .show()
)

+----------+---------------------+-------------------+-------------------+
|  batch_id|year_level_month_rows|min_attendance_rate|max_attendance_rate|
+----------+---------------------+-------------------+-------------------+
|2025-01-15|                  156|             0.7419|             0.8602|
|2026-01-15|                  156|             0.7501|             0.8634|
+----------+---------------------+-------------------+-------------------+



### gold.assessment_by_school, gold.assessment_by_domain

In [0]:
# gold.assessment_by_school

assessment_df = (
    spark.table(f"{catalog}.silver.assessment_results")
    .filter(F.col("batch_id") == batch_id))
schools_df = (
    spark.table(f"{catalog}.silver.schools")
    .filter(F.col("batch_id") == batch_id))

invalid_assessment_records_df = (
    spark.table(f"{catalog}.qa.dq_failed_records")
    .filter(
        (F.col("run_id") == run_id)
        & (F.col("target_table") == "silver.assessment_results")
        & (F.col("rule_id").isin(["DQ006", "DQ007", "DQ008"]))
    )
    .select("bronze_record_id")
    .distinct()
)

valid_assessment_df = (
    assessment_df
    .join(
        invalid_assessment_records_df,
        on="bronze_record_id",
        how="left_anti"
    )
)

assessment_by_school_df = (
    valid_assessment_df.alias("a")
    .join(
        schools_df.select(
            "batch_id",
            "school_id",
            "school_name",
            "region",
            "school_type",
            "status"
        ).alias("s"),
        on=["batch_id", "school_id"],
        how="left"
    )
    .groupBy(
        "batch_id",
        "assessment_year",
        "school_id",
        "school_name",
        "region",
        "school_type"
    )
    .agg(
        F.countDistinct("assessment_id").alias("assessment_count"),
        F.countDistinct("student_id").alias("student_count"),
        F.round(F.avg("score"), 2).alias("average_score"),
        F.round(F.expr("percentile_approx(score, 0.5)"), 2).alias("median_score"),
        F.min("score").alias("min_score"),
        F.max("score").alias("max_score")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

target_path = f"{lake_root}/gold/assessment_by_school"

merge_condition = """
target.batch_id = source.batch_id
AND target.assessment_year = source.assessment_year
AND target.school_id = source.school_id
"""

write_delta_table(
    df=assessment_by_school_df,
    table_name="gold.assessment_by_school",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.assessment_by_school")
    .orderBy("batch_id", "assessment_year", "school_id")
    .limit(50)
)

batch_id,assessment_year,school_id,school_name,region,school_type,assessment_count,student_count,average_score,median_score,min_score,max_score,gold_load_timestamp
2025-01-15,2024,SCH001,ACT Education School 001,North Canberra,High School,600,200,516.1,514,413,620,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH002,ACT Education School 002,North Canberra,Primary,630,210,367.98,368,250,518,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH003,ACT Education School 003,North Canberra,Primary,567,189,372.14,370,250,520,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH004,ACT Education School 004,North Canberra,Primary,513,171,386.18,390,250,515,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH005,ACT Education School 005,Tuggeranong,High School,537,179,513.27,514,412,619,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH006,ACT Education School 006,Weston Creek,Primary,609,203,369.02,367,250,514,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH007,ACT Education School 007,Tuggeranong,Primary,600,200,375.59,371,250,518,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH008,ACT Education School 008,Weston Creek,Primary,591,197,381.87,382,250,516,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH009,ACT Education School 009,Belconnen,Primary,624,208,380.59,382,250,520,2026-06-07T02:18:22.574Z
2025-01-15,2024,SCH010,ACT Education School 010,Woden,College,567,189,590.41,591,510,669,2026-06-07T02:18:22.574Z


In [0]:
(
    spark.table(f"{catalog}.gold.assessment_by_school")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("school_assessment_rows"),
        F.min("average_score").alias("min_average_score"),
        F.max("average_score").alias("max_average_score")
    )
    .orderBy("batch_id")
    .show()
)

+----------+----------------------+-----------------+-----------------+
|  batch_id|school_assessment_rows|min_average_score|max_average_score|
+----------+----------------------+-----------------+-----------------+
|2025-01-15|                    50|           360.58|           591.61|
|2026-01-15|                    48|           376.03|            597.8|
+----------+----------------------+-----------------+-----------------+



In [0]:
# gold.assessment_by_domain

assessment_df = (
    spark.table(f"{catalog}.silver.assessment_results")
    .filter(F.col("batch_id") == batch_id)
)

invalid_assessment_records_df = (
    spark.table(f"{catalog}.qa.dq_failed_records")
    .filter(
        (F.col("run_id") == run_id)
        & (F.col("target_table") == "silver.assessment_results")
        & (F.col("rule_id").isin(["DQ006", "DQ007", "DQ008"]))
    )
    .select("bronze_record_id")
    .distinct()
)

valid_assessment_df = (
    assessment_df
    .join(
        invalid_assessment_records_df,
        on="bronze_record_id",
        how="left_anti"
    )
)

assessment_by_domain_df = (
    valid_assessment_df
    .groupBy(
        "batch_id",
        "assessment_year",
        "domain",
        "proficiency_band"
    )
    .agg(
        F.countDistinct("assessment_id").alias("assessment_count"),
        F.countDistinct("student_id").alias("student_count"),
        F.round(F.avg("score"), 2).alias("average_score"),
        F.round(F.expr("percentile_approx(score, 0.5)"), 2).alias("median_score"),
        F.min("score").alias("min_score"),
        F.max("score").alias("max_score")
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

target_path = f"{lake_root}/gold/assessment_by_domain"

merge_condition = """
target.batch_id = source.batch_id
AND target.assessment_year = source.assessment_year
AND target.domain = source.domain
AND target.proficiency_band = source.proficiency_band
"""

write_delta_table(
    df=assessment_by_domain_df,
    table_name="gold.assessment_by_domain",
    target_path=target_path,
    merge_condition=merge_condition
)

display(
    spark.table(f"{catalog}.gold.assessment_by_domain")
    .orderBy("batch_id", "assessment_year", "domain", "proficiency_band")
)

batch_id,assessment_year,domain,proficiency_band,assessment_count,student_count,average_score,median_score,min_score,max_score,gold_load_timestamp
2025-01-15,2024,Numeracy,High,1786,1786,594.3,591,550,660,2026-06-07T02:19:59.339Z
2025-01-15,2024,Numeracy,Low,4165,4165,338.21,342,250,399,2026-06-07T02:19:59.339Z
2025-01-15,2024,Numeracy,Medium,4049,4049,467.58,463,400,549,2026-06-07T02:19:59.339Z
2025-01-15,2024,Reading,High,2060,2060,598.33,594,550,670,2026-06-07T02:19:59.339Z
2025-01-15,2024,Reading,Low,3790,3790,341.64,347,250,399,2026-06-07T02:19:59.339Z
2025-01-15,2024,Reading,Medium,4150,4150,465.54,460,400,549,2026-06-07T02:19:59.339Z
2025-01-15,2024,Writing,High,1617,1617,593.04,590,550,655,2026-06-07T02:19:59.339Z
2025-01-15,2024,Writing,Low,4332,4332,336.5,341,250,399,2026-06-07T02:19:59.339Z
2025-01-15,2024,Writing,Medium,4051,4051,467.22,462,400,549,2026-06-07T02:19:59.339Z
2026-01-15,2025,Numeracy,High,1309,1309,593.39,590,550,660,2026-06-07T02:19:59.339Z


In [0]:
(
    spark.table(f"{catalog}.gold.assessment_by_domain")
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("domain_band_rows"),
        F.min("average_score").alias("min_average_score"),
        F.max("average_score").alias("max_average_score")
    )
    .orderBy("batch_id")
    .show()
)

+----------+----------------+-----------------+-----------------+
|  batch_id|domain_band_rows|min_average_score|max_average_score|
+----------+----------------+-----------------+-----------------+
|2025-01-15|               9|            336.5|           598.33|
|2026-01-15|               9|           343.53|           596.28|
+----------+----------------+-----------------+-----------------+



In [0]:
# Validation

gold_tables = [
    "data_quality_summary",
    "data_quality_rule_detail",
    "attendance_by_school_month",
    "attendance_by_year_level",
    "assessment_by_school",
    "assessment_by_domain"
]

gold_validation_dfs = []

for table_name in gold_tables:
    df = spark.table(f"{catalog}.gold.{table_name}")

    validation_df = (
        df.groupBy("batch_id")
        .agg(F.count("*").alias("row_count"))
        .withColumn("gold_table", F.lit(table_name))
        .select("gold_table", "batch_id", "row_count")
    )

    gold_validation_dfs.append(validation_df)

gold_output_validation_df = reduce(
    lambda df1, df2: df1.unionByName(df2),
    gold_validation_dfs
)

display(
    gold_output_validation_df
    .orderBy("gold_table", "batch_id")
)

gold_table,batch_id,row_count
assessment_by_domain,2025-01-15,9
assessment_by_domain,2026-01-15,9
assessment_by_school,2025-01-15,50
assessment_by_school,2026-01-15,48
attendance_by_school_month,2025-01-15,600
attendance_by_school_month,2026-01-15,576
attendance_by_year_level,2025-01-15,156
attendance_by_year_level,2026-01-15,156
data_quality_rule_detail,2025-01-15,11
data_quality_rule_detail,2026-01-15,11
